In [1]:
import ast

s = '''
describe_point(p)
match i:
    case Enum.m(): 
        return "One" 
'''

print(ast.dump(ast.parse(s).body[0]))

Expr(value=Call(func=Name(id='describe_point', ctx=Load()), args=[Name(id='p', ctx=Load())]))


In [2]:
from guppylang import guppy

@guppy
def main() -> int:
    if True:
        return 1

main.check()

In [3]:
# Example: linear variable used in multiple match arms (should error if not allowed)
from guppylang import guppy
from guppylang.std.quantum import qubit, measure, owned, h

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def fun() -> Point:
    return Point(qubit(), 4)

@guppy
def main(p: Point) -> None:
    match fun():
        case _:
            pass

    h(p.x)

main.check()


Error: Drop violation (at <In[3]>:16:10)
   | 
14 | @guppy
15 | def main(p: Point) -> None:
16 |     match fun():
   |           ^^^^^ Expression with non-droppable type `Point` is leaked

Help: Consider assigning this value to a local variable

Guppy compilation failed due to 1 previous error


In [ ]:
# Example: linear variable used in multiple match arms (should error if not allowed)
from guppylang import guppy
from guppylang.std.quantum import qubit, measure, owned, h

@guppy.enum
class Point:
    x = {"n": qubit}

@guppy
def fun() -> Point:
    return Point.x(qubit())

@guppy
def main(p: Point) -> None:
    match fun():
        case _:
            pass

main.check()


Error: Drop violation (at <In[4]>:15:10)
   | 
13 | @guppy
14 | def main(p: Point) -> None:
15 |     match fun():
   |           ^^^^^ Expression with non-droppable type `Point` is leaked

Help: Consider assigning this value to a local variable

Guppy compilation failed due to 1 previous error


In [ ]:
# Example: linear variable used in multiple match arms (should error if not allowed)
from guppylang import guppy
from guppylang.std.quantum import qubit, measure, h

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def fun() -> Point:
    return Point(qubit(), 4)

@guppy
def describe_point(point: Point)-> None:
    pass

@guppy
def main(p: Point) -> None:

    describe_point(fun())

main.check()


In [ ]:
# Example: linear variable used in multiple match arms (should error if not allowed)
from guppylang import guppy
from guppylang.std.quantum import qubit, measure

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def fun() -> Point:
    return Point(qubit(), 4)

@guppy
def describe_point(point: Point)-> None:
    pass

@guppy
def not_consume(p: Point) -> Point:
    return Point(qubit(), 4)

@guppy
def main(p: Point @owned) -> None:
    match not_consume(p):
        case Point(_, 4):
            pass
    # describe_point(fun())

main.check()
